# core

> `astream` and `ClaudeRun`: stateless completions through the installed Claude Code

In [ ]:
#| default_exp core

`astream(msgs, ...)` creates a `ClaudeRun` for one stateless completion through your installed `claude`. It uses Claude Code's login and subscription. Supply the complete conversation as `aidialog.msg_parts.Msg` objects on every request. FastClaude writes the history as a native transcript with `fastclaude.session` and starts a fresh process to resume it.

Your application executes the tools it supplies. When Claude requests one, the run ends with a `ToolUse`. Execute the tool and append its `ToolResult` to the history for the next request. FastClaude marks the pending call in the transcript. Its `fastclaude.protocol` bridge returns your result when Claude resumes that call.

Iterate the run for raw stream-json events. `run.messages` contains the generated `Msg` trace with thinking signatures intact and your original tool names. `run.result` contains the terminal result. Append the trace to your history for the next request.

Use `run.interrupt()` to end the current turn. Close the run to stop its process and remove its temporary transcript. Runs use a dedicated XDG cache work directory by default. Pass `cwd=` to use a project's context or `native_tools=` to enable Claude Code tools such as `WebSearch`.


In [ ]:
#| export
import asyncio, json, os, shutil, uuid
from contextlib import suppress
from fastcore.utils import *
from fastcore.meta import delegates
from fastcore.xdg import xdg_cache_home
from fastllm.anthropic import denorm_msgs, norm_parts
from aidialog.msg_parts import Msg, Text, ToolUse, ToolResult, Media
from fastclaude.session import *
from fastclaude.protocol import *

In [ ]:
from fastcore.test import *
import stat, sys, tempfile, textwrap

## The work dir

Claude Code looks for a resumed session under `~/.claude/projects`, in a folder derived from the process's working directory. FastClaude must write its transcript there for `--resume` to find it. The working directory also determines which project settings and `CLAUDE.md` Claude Code reads.

`work_dir()` creates a dedicated `fastclaude` directory under the XDG cache directory. All default runs share it. They don't inherit the host process's current project or add transcripts to that project's session list. This separates the runs from real projects without isolating your Claude Code login or user settings. Pass `cwd=` when you want a particular project's context.

In [ ]:
#| export
MCP_SERVER = 'fastclaude'
MCP_PREFIX = f'mcp__{MCP_SERVER}__'
SERVER_TOOLS = ('WebSearch','WebFetch')

def work_dir():
    "Create and return the shared FastClaude work directory under the XDG cache directory"
    p = xdg_cache_home()/'fastclaude'
    p.mkdir(parents=True, exist_ok=True)
    return p

## Compiling history

`compile_msgs` prepares the history for a new process. You can edit, hide, rewind, or replace earlier messages before each request. `denorm_msgs` converts the supplied `Msg` objects to Anthropic-style messages. `prefix_tools` adds `mcp__fastclaude__` to external tool names. It leaves existing `mcp__` names and the native `WebSearch` and `WebFetch` names unchanged.

The final message determines how the run starts. A user message with text or media supplies the live prompt. Everything before it becomes the transcript. A final tool result instead asks Claude to continue its pending call. FastClaude resumes that call without sending a new prompt.

Claude Code re-invokes only the last deferred call. For multiple trailing results, `compile_msgs` includes all but the last in the history. It returns the last call and result separately for `_spawn` to prepare the continuation.


In [ ]:
#| export
def compile_msgs(
    msgs, # Complete history as `Msg`s, ending with a user prompt or with the tool results Claude asked for
):
    """Return `(history, prompt, deferred)` for a new run.

    `history` contains wire messages for the transcript. `prompt` is the live turn's content, or None for a continuation.
    `deferred` is the pending `(tool_use, tool_result)` pair, or None for a live prompt.
    """
    msgs = listify(msgs)
    if not msgs: raise ValueError('empty message history')
    den = prefix_tools(denorm_msgs(msgs), MCP_PREFIX, skip=SERVER_TOOLS)
    lc = den[-1]['content']
    if msgs[-1].role=='user' and any(isinstance(p, (Text,Media)) for p in msgs[-1].content): return den[:-1], lc, None
    if den[-1]['role']=='user' and isinstance(lc, list) and lc and all(b.get('type')=='tool_result' for b in lc):
        tus = {b['id']: b for m in den for b in (m['content'] if isinstance(m['content'], list) else []) if b.get('type')=='tool_use'}
        pend = [tus.get(b['tool_use_id']) for b in lc]
        if None in pend: raise ValueError('a trailing tool result answers no tool_use in the history')
        hist = den[:-1] + ([dict(role='user', content=lc[:-1])] if len(lc)>1 else [])
        return hist, None, (pend[-1], lc[-1])
    raise ValueError('history must end with a user prompt or tool results')

In [ ]:
prior = Msg('user', [Text('Measure the flux please.')])
call = Msg('assistant', [Text('Checking.'), ToolUse(id='t1', name='flux_meter', arguments={})])
result = Msg('tool', [ToolResult(id='t1', name='flux_meter', text='flux: 41.7 kf')])
prompt = Msg('user', [Text('And in gauss?')])
h = [prior,call,result,prompt]
h

[Msg(role='user', content=[Text(raw=None, cache_control=None, text='Measure the flux please.', citations=None)], raw=None),
 Msg(role='assistant', content=[Text(raw=None, cache_control=None, text='Checking.', citations=None), ToolUse(raw=None, cache_control=None, id='t1', name='flux_meter', arguments={}, server=False, text=None)], raw=None),
 Msg(role='tool', content=[ToolResult(raw=None, cache_control=None, id='t1', name='flux_meter', arguments={}, server=False, text='flux: 41.7 kf')], raw=None),
 Msg(role='user', content=[Text(raw=None, cache_control=None, text='And in gauss?', citations=None)], raw=None)]

`compile_msgs(h)` returns the first three messages as history and the last user's content as the live prompt. The history uses the qualified external tool name. This function prepares messages without writing a transcript.

In [ ]:
hist,prompt,deferred = compile_msgs(h)
test_eq((len(hist), deferred), (3, None))
test_eq(hist[1]['content'][1]['name'], 'mcp__fastclaude__flux_meter')
test_eq(prompt, [dict(type='text', text='And in gauss?')])
hist[1]

{'role': 'assistant',
 'content': [{'type': 'text', 'text': 'Checking.'},
  {'type': 'tool_use',
   'id': 't1',
   'name': 'mcp__fastclaude__flux_meter',
   'input': {}}]}

Ending the same history at its tool result requests a continuation. `compile_msgs(h[:3])` returns no live prompt. The separate `(tool_use, tool_result)` pair identifies the pending call by its qualified name. `_spawn` will mark that call with `mk_deferred` and give the result to the bridge.

In [ ]:
chist,cprompt,(ctu,ctr) = compile_msgs(h[:3])
test_eq((len(chist), cprompt), (2, None))
test_eq((ctu['id'], ctu['name']), ('t1', 'mcp__fastclaude__flux_meter'))
ctr

{'type': 'tool_result', 'tool_use_id': 't1', 'content': 'flux: 41.7 kf'}

In this example, Claude requested two measurements. The history contains the result for `a`. The continuation will supply the result for `b` when Claude re-invokes that call.

In [ ]:
h2 = h[:1] + [Msg('assistant', [ToolUse(id='a', name='flux_meter', arguments={}), ToolUse(id='b', name='flux_meter', arguments=dict(unit='gauss'))]),
    Msg('tool', [ToolResult(id='a', name='flux_meter', text='flux: 41.7 kf'), ToolResult(id='b', name='flux_meter', text='flux: 41.7 gauss')])]
h2hist,_,h2d = compile_msgs(h2)
test_eq(h2hist[-1]['content'][0]['tool_use_id'], 'a')
test_eq(h2d[0]['id'], 'b')
h2hist[-1]

{'role': 'user',
 'content': [{'type': 'tool_result',
   'tool_use_id': 'a',
   'content': 'flux: 41.7 kf'}]}

`compile_msgs` raises `ValueError` for an empty history, an invalid final message, or a trailing result with no matching `tool_use`. These checks run before FastClaude starts a process.

In [ ]:
with expect_fail(ValueError, contains='user prompt or tool results'): compile_msgs(h[:2])
with expect_fail(ValueError, contains='no tool_use'): compile_msgs(h[:1] + [Msg('tool', [ToolResult(id='zz', name='f', text='x')])])
with expect_fail(ValueError, contains='empty'): compile_msgs([])

## The command and environment

`claude_cmd` uses your installed `claude` and preserves its config and login. It does not change `CLAUDE_CONFIG_DIR`. Isolating that directory would also isolate the login needed for subscription access.

The command always includes `--strict-mcp-config`. Claude Code cannot use unrelated MCP servers from your configuration, even when the run supplies no tools. Pass `mcp_config` to add external servers, such as a stdio process. When you supply tool schemas, the command adds a private SDK server entry to `--mcp-config` for FastClaude's bridge.

Built-in tools are off by default. `--tools` always contains an explicit list from `native_tools`, including an empty list when you enable none. `setting_sources=None` keeps the CLI's default settings sources. Use `['project']` for project settings alone or `()` to load none.

`--resume=<id>` uses an equals sign. A session id beginning with a dash cannot become a separate command option.

`claude_env` copies the parent environment with two changes. It empties `ANTHROPIC_API_KEY` to prevent billing through an inherited API key. It removes `CLAUDECODE` to avoid the CLI's nested-session check. `ClaudeRun(env=...)` can override these defaults.

In [ ]:
#| export
def claude_cmd(
    model=None, # Model alias or full name; None uses the user's default
    resume=None, # Session id to resume, i.e. the transcript just written
    system=None, # System prompt; None keeps Claude Code's own
    tools=False, # Offer the private SDK MCP server?
    native_tools=(), # Built-in Claude Code tools to enable, e.g. 'WebSearch'
    allowed=(), # `--allowedTools` entries, e.g. qualified callable names
    append_system=None, # Text appended to Claude Code's own system prompt, which stays
    setting_sources=None, # Settings that load, e.g. ['project']; () loads none; None keeps the CLI default (all)
    mcp_config=None, # Extra MCP server entries, e.g. `dict(clikernel=dict(type='stdio', command=...))`
    max_turns=None, # Bound on agent turns; None is unbounded
    max_budget=None, # Max USD for the run; None is unbounded
    permission_mode=None, # e.g. 'bypassPermissions'; None keeps the CLI default
    thinking=None, # 'adaptive' or 'disabled'; None keeps the CLI default
    effort=None, # Thinking effort: 'low', 'medium', or 'high'
    claude_path=None, # Explicit claude executable; found on PATH if None
):
    "Build argv for one headless stream-json Claude Code run"
    c = [str(claude_path or shutil.which('claude') or 'claude'), '--output-format','stream-json',
        '--input-format','stream-json', '--verbose', '--include-partial-messages']
    if model: c += ['--model', model]
    if system is not None: c += ['--system-prompt', system]
    if append_system: c += ['--append-system-prompt', append_system]
    if resume: c += [f'--resume={resume}']
    servers = dict(mcp_config or {})
    if tools: servers[MCP_SERVER] = dict(type='sdk', name=MCP_SERVER)
    if servers: c += ['--mcp-config', json.dumps(dict(mcpServers=servers))]
    c.append('--strict-mcp-config')
    c += ['--tools', ','.join(native_tools)]
    if allowed: c += ['--allowedTools', ','.join(allowed)]
    if setting_sources is not None: c.append(f"--setting-sources={','.join(setting_sources)}")
    for f,v in (('--max-turns',max_turns), ('--max-budget-usd',max_budget), ('--permission-mode',permission_mode), ('--thinking',thinking), ('--effort',effort)):
        if v is not None: c += [f, str(v)]
    return c

def claude_env():
    "Copy the environment with an empty `ANTHROPIC_API_KEY` and no `CLAUDECODE`"
    env = dict(os.environ, ANTHROPIC_API_KEY='')
    env.pop('CLAUDECODE', None)
    return env

In [ ]:
c = claude_cmd('sonnet', resume='-abc', tools=True, allowed=['mcp__fastclaude__flux_meter'])
test('--resume=-abc', c, in_)
test_eq(c[c.index('--tools')+1], '')
test('--strict-mcp-config', c, in_)
test('--strict-mcp-config', claude_cmd('sonnet'), in_)
test_eq(json.loads(c[c.index('--mcp-config')+1]), dict(mcpServers=dict(fastclaude=dict(type='sdk', name='fastclaude'))))
env = claude_env()
test_eq(env['ANTHROPIC_API_KEY'], '')
assert 'CLAUDECODE' not in env
c[1:]

['--output-format',
 'stream-json',
 '--input-format',
 'stream-json',
 '--verbose',
 '--include-partial-messages',
 '--model',
 'sonnet',
 '--resume=-abc',
 '--mcp-config',
 '{"mcpServers": {"fastclaude": {"type": "sdk", "name": "fastclaude"}}}',
 '--strict-mcp-config',
 '--tools',
 '',
 '--allowedTools',
 'mcp__fastclaude__flux_meter']

Optional arguments add flags only when you supply a value. Here we add an external MCP server, select project settings, and limit the run to eight agent turns.

In [ ]:
c2 = claude_cmd(mcp_config=dict(clikernel=dict(type='stdio', command='clikernel-mcp')), setting_sources=['project'], max_turns=8)
test_eq(json.loads(c2[c2.index('--mcp-config')+1])['mcpServers']['clikernel']['type'], 'stdio')
test('--setting-sources=project', c2, in_)
test_eq(c2[c2.index('--max-turns')+1], '8')
assert '--max-turns' not in claude_cmd()
c2[1:]

['--output-format',
 'stream-json',
 '--input-format',
 'stream-json',
 '--verbose',
 '--include-partial-messages',
 '--mcp-config',
 '{"mcpServers": {"clikernel": {"type": "stdio", "command": "clikernel-mcp"}}}',
 '--strict-mcp-config',
 '--tools',
 '',
 '--setting-sources=project',
 '--max-turns',
 '8']

## The run

A `ClaudeRun` starts when you iterate it. It prepares the transcript, starts Claude Code, and completes the control handshake before sending any live prompt. A continuation resumes from the transcript without a prompt. Iteration yields every non-control event unchanged.

Each run handles one completion. It does not keep a process for the next request. When Claude asks for an advertised tool, the run ends before execution. Your application executes the `ToolUse` from `run.messages` and supplies the result in a new request.


In [ ]:
#| export
class ClaudeRun:
    "Manage one Claude Code process for a stateless completion with streamed events and a reusable message trace"
    @delegates(claude_cmd, but=['model','resume','tools','native_tools','allowed'])
    def __init__(self,
        msgs, # Complete history as `Msg`s, ending with a user prompt or the tool results Claude asked for
        model='sonnet', # Model alias or full name
        tools=None, # Tool schemas to advertise, in either `tool_spec` form; the caller executes
        cwd=None, # Project directory for the run; the shared `work_dir()` if None
        native_tools=(), # Built-in Claude Code tools to enable, e.g. 'WebSearch'
        allowed=(), # Extra `--allowedTools` entries beyond the advertised schemas
        env=None, # Extra child environment variables, merged over `claude_env()`
        **kwargs, # Passed to `claude_cmd`, e.g. `system`, `setting_sources`, `mcp_config`
    ):
        store_attr('msgs,model,tools,cwd,native_tools,allowed,env')
        self.cmd_kwargs = kwargs
        self.messages,self.result,self.proc,self.proto = [],None,None,None
        self._names,self._closed,self._spath,self._deferred = {},False,None,None

    def __aiter__(self):
        if not hasattr(self, '_it'): self._it = self._run()
        return self._it

@delegates(ClaudeRun)
def astream(msgs, **kwargs):
    "Create a `ClaudeRun` for one stateless completion and iterate it for raw events"
    return ClaudeRun(msgs, **kwargs)

`_spawn` writes the history with a fresh random session id. Concurrent requests can use identical histories without sharing a transcript. Prompt caching depends on the request content, not reuse of the session id.

For a continuation, `_spawn` adds a `mk_deferred` record for the pending call. It converts your result with `tool_reply` and gives that reply to `ClaudeProto` as `held`. Claude Code will request it on resume. A first request containing only a user prompt needs no transcript and no `--resume` option.

The subprocess uses piped standard input and output. `_spawn` raises `FileNotFoundError` if it cannot find the executable and removes any transcript it wrote.


In [ ]:
#| export
@patch
async def _spawn(self:ClaudeRun):
    """Write the transcript and start Claude Code.

    Return the live prompt content, or None for a continuation.
    """
    self.cwd = Path(self.cwd).expanduser() if self.cwd else work_dir()
    hist,prompt,deferred = compile_msgs(self.msgs)
    sid = str(uuid.uuid4())
    recs = msgs2recs(hist, key=sid, cwd=self.cwd)
    held = None
    if deferred:
        tu,tr = deferred
        recs.append(mk_deferred(tu, cwd=self.cwd))
        held,self._deferred = tool_reply(tr.get('content',''), tr.get('is_error', False)),tu['id']
    if recs:
        save_sess(recs, sid, self.cwd)
        self._spath = sess_file(sid, self.cwd)
    schemas = mk_tools(self.tools or [])
    allowed = [MCP_PREFIX+s['name'] for s in schemas] + list(self.native_tools) + list(self.allowed)
    argv = claude_cmd(self.model, resume=sid if recs else None, tools=bool(schemas),
        native_tools=self.native_tools, allowed=allowed, **self.cmd_kwargs)
    if not shutil.which(argv[0]):
        await self.aclose()
        raise FileNotFoundError(f"claude executable not found: {argv[0]}")
    self.proc = await asyncio.create_subprocess_exec(*argv, stdin=asyncio.subprocess.PIPE,
        stdout=asyncio.subprocess.PIPE, limit=2**25, cwd=self.cwd, env=dict(claude_env(), **(self.env or {})))
    self.proto = ClaudeProto(self.proc, tools=self.tools, held=held, server=MCP_SERVER)
    return prompt

`_track` builds `run.messages` from full message events, not partial streaming updates. Each full `assistant` event becomes a `Msg` through `norm_parts`. These events contain individual content blocks, including signed thinking. `unqual` removes the `mcp__fastclaude__` prefix from tool names in the trace.

A `user` event containing tool results becomes a `Msg` with role `tool`. On a continuation, `_track` skips the event containing the result you supplied. Your history already contains it. Native tool results, such as those from `WebSearch`, still enter the trace. `_track` stores the terminal `result` event in `run.result`.

None of this changes the raw events that iteration yields.


In [ ]:
#| export
def unqual(nm):
    "Remove the `mcp__fastclaude__` prefix from `nm` when present"
    return nm[len(MCP_PREFIX):] if nm and nm.startswith(MCP_PREFIX) else nm

def _flat(c): return c if isinstance(c, str) else '\n'.join(b.get('text','') for b in c if b.get('type')=='text')

@patch
def _track(self:ClaudeRun, m):
    "Update `.messages` or `.result` from one raw event"
    t,c = m.get('type'), nested_idx(m, 'message', 'content')
    if t=='assistant' and isinstance(c, list):
        parts = norm_parts(m['message'])
        for p in parts:
            if isinstance(p, ToolUse): p.name = self._names[p.id] = unqual(p.name)
        self.messages.append(Msg('assistant', parts))
    elif t=='user' and isinstance(c, list) and c and all(b.get('type')=='tool_result' for b in c):
        if any(b.get('tool_use_id')==self._deferred for b in c): return
        self.messages.append(Msg('tool', [ToolResult(id=b.get('tool_use_id'), name=self._names.get(b.get('tool_use_id')),
            text=_flat(b.get('content',''))) for b in c]))
    elif t=='result': self.result = m

`_kick` waits for the initialization handshake before sending the live prompt. `_run` starts it as a separate task while reading protocol events. That reader must be active to handle the handshake response. A continuation completes the handshake without sending a user message.

In [ ]:
#| export
@patch
async def _kick(self:ClaudeRun, prompt):
    "Complete the handshake and send the live prompt if present"
    await self.proto.initialize()
    if prompt is not None: await self.proto.send(dict(type='user', message=dict(role='user', content=prompt)))

`_run` updates the trace before yielding each event. After yielding the terminal result, it ends iteration and calls `aclose`. Its `finally` block also runs on errors, cancellation during iteration, or explicit closure of the iterator. If you stop consuming early with `break`, call `await run.aclose()` rather than relying on generator finalization.

In [ ]:
#| export
@patch
async def _run(self:ClaudeRun):
    "Start Claude Code and yield raw events through its terminal result"
    prompt = await self._spawn()
    kick = asyncio.create_task(self._kick(prompt))
    try:
        async for m in self.proto.events():
            self._track(m)
            yield m
            if m.get('type')=='result': break
    finally:
        kick.cancel()
        await self.aclose()

`interrupt` sends Claude Code's native control request to end the current turn. Keep consuming the event stream to receive the remaining output and terminal result.

In [ ]:
#| export
@patch
async def interrupt(self:ClaudeRun, timeout=30):
    "Ask Claude Code to end the turn while keeping the stream open for remaining events"
    return await self.proto.interrupt(timeout)

Hosts often handle Ctrl-C by cancelling the task that consumes the run. Cleanup must still stop the child process and remove the temporary transcript. It does not read more output or finish the trace. Use `interrupt` while continuing iteration when you need the final events.

`_cleanup` closes standard input to ask Claude Code to exit. It then calls `_reap_process` to wait for the process and stop it forcibly if necessary.

`_wait_process` returns `True` if the process exits within the timeout and `False` otherwise. Its default timeout is five seconds.

In [ ]:
#| export
async def _wait_process(proc, timeout=5):
    try:
        await asyncio.wait_for(proc.wait(), timeout)
        return True
    except (TimeoutError, asyncio.TimeoutError): return False

`_reap_process` waits up to five seconds for normal exit, then sends terminate and waits another five seconds. If the process still hasn't exited, it sends kill and waits without a timeout. It returns as soon as a wait succeeds.

In [ ]:
#| export
async def _reap_process(proc):
    if await _wait_process(proc): return
    with suppress(ProcessLookupError): proc.terminate()
    if await _wait_process(proc): return
    with suppress(ProcessLookupError): proc.kill()
    with suppress(Exception): await proc.wait()

After stopping the process, `_cleanup` cancels protocol work and removes the exact transcript this run wrote. It does not delete other sessions.

In [ ]:
#| export
@patch
async def _cleanup(self:ClaudeRun):
    p = self.proc
    if p and p.returncode is None:
        with suppress(Exception): p.stdin.close()
        await _reap_process(p)
    if self.proto: await self.proto.aclose()
    if self._spath: Path(self._spath).unlink(missing_ok=True)

`aclose` starts cleanup once, even if you call it repeatedly. `asyncio.shield` lets cleanup continue if the awaiting task receives another cancellation. That task can stop waiting before cleanup finishes.

In [ ]:
#| export
@patch
async def aclose(self:ClaudeRun):
    """Close stdin, stop the process, and remove the transcript.

    Start cleanup once and shield it from cancellation.
    """
    if self._closed: return
    self._closed = True
    await asyncio.shield(asyncio.create_task(self._cleanup()))

## A scripted run

This test runs FastClaude against a script instead of a model. The script acknowledges control requests, echoes the user prompt as an assistant message, and emits a terminal result. It exercises handshake ordering, trace collection, completion, and transcript removal without spending tokens.

In [ ]:
# chkstyle: ignore-node
fake_src = textwrap.dedent('''
    #!/usr/bin/env python3
    import sys, json
    def w(o): sys.stdout.write(json.dumps(o)+'\\n'); sys.stdout.flush()
    for line in sys.stdin:
        m = json.loads(line)
        if m.get('type')=='control_request':
            w(dict(type='control_response', response=dict(subtype='success', request_id=m['request_id'], response={})))
        elif m.get('type')=='user':
            c = m['message']['content']
            txt = 'echo: '+(c if isinstance(c, str) else c[0].get('text',''))
            w(dict(type='assistant', message=dict(role='assistant', content=[dict(type='text', text=txt)])))
            w(dict(type='result', subtype='success', result=txt))
    ''').strip()

We make the script executable and pass its path as `claude_path`. FastClaude still starts a real subprocess and communicates through pipes. The script gives repeatable results without authentication or a model connection.

In [ ]:
fake_cc = Path(tempfile.mkdtemp())/'claude'
fake_cc.write_text(fake_src+'\n')
fake_cc.chmod(fake_cc.stat().st_mode | stat.S_IXUSR)

In [ ]:
scratch = Path(tempfile.mkdtemp())
run = astream(h, claude_path=fake_cc, cwd=scratch)
got = [m async for m in run]
test_eq(run.result['result'], 'echo: And in gauss?')
test_eq([m.role for m in run.messages], ['assistant'])
test_eq(run.messages[0].content[0].text, 'echo: And in gauss?')
test_eq(run._spath.exists(), False)
with expect_fail(FileNotFoundError, contains='not found'): [m async for m in astream(h, claude_path=scratch/'missing', cwd=scratch)]
[m['type'] for m in got]

['assistant', 'result']

## Live runs

The following examples use your authenticated Claude Code CLI and spend tokens. Their `eval: false` directives exclude them from automated tests.

We'll ask Claude to use `flux_meter`. Passing this function in `tools` advertises its schema. FastClaude never calls the function body.


In [ ]:
async def flux_meter(unit:str='kf') -> str:
    "Read the flux."
    return f'flux: 41.7 {unit}'

When Claude requests `flux_meter`, the `PreToolUse` hook defers the call and ends the turn. We consume the run until its process exits. No tool has executed yet.

In [ ]:
#| eval: false
lmsgs = [Msg('user', [Text('Use the flux_meter tool with unit="gauss", then reply with exactly the tool output.')])]
lrun = astream(lmsgs, model='claude-sonnet-5', tools=[flux_meter])
levs = [m async for m in lrun]
L(m.get('type') for m in levs).unique()

`lrun.messages` ends with the pending `ToolUse`, using the original name `flux_meter`. The run has a successful terminal result and no running process.

In [ ]:
#| eval: false
ltu = lrun.messages[-1].content[-1]
test_eq((type(ltu), ltu.name, ltu.arguments), (ToolUse, 'flux_meter', dict(unit='gauss')))
test_eq(lrun.result['subtype'], 'success')
[(m.role, [type(p).__name__ for p in m.content]) for m in lrun.messages]

Execute the requested call in your application, then append the trace and its `ToolResult` to the history. Pass that history to `astream` for the continuation.

Claude Code resumes the pending call and collects the result from the bridge without another live prompt. FastClaude does not execute the function again. The new trace contains Claude's continuation without duplicating the result you supplied.


In [ ]:
#| eval: false
out = await flux_meter(**ltu.arguments)
cmsgs = lmsgs + lrun.messages + [Msg('tool', [ToolResult(id=ltu.id, name=ltu.name, text=out)])]
crun = astream(cmsgs, model='claude-sonnet-5', tools=[flux_meter])
cevs = [m async for m in crun]
test_eq([m.role for m in crun.messages], ['assistant'])
crun.result['result']


'flux: 41.7 gauss'

You can interrupt Claude during generation. This example sends `interrupt` after twenty streaming events and continues reading to the end. Claude reports `is_error=True` and `subtype='error_during_execution'` in the stored result. Those fields alone do not distinguish interruption from other execution failures.

FastClaude cannot cancel work in your external tool loop. Your application must cancel its own tool execution.

In [ ]:
#| eval: false
irun = astream([Msg('user', [Text('Write a 2000 word essay on the history of magnetometry. Do not stop early.')])])
n = 0
async for m in irun:
    if m.get('type')=='stream_event' and (n := n+1)==20: asyncio.ensure_future(irun.interrupt())
test_eq([m.role for m in irun.messages], ['assistant'])
{k: irun.result.get(k) for k in ('subtype','is_error')}


{'subtype': 'error_during_execution', 'is_error': True}

For the next prompt, include the completed exchange in the history. FastClaude writes the signed thinking and tool exchange into the new transcript. Claude can refer to the earlier measurement without executing the tool again.


In [ ]:
#| eval: false
qmsgs = cmsgs + crun.messages + [Msg('user', [Text('What unit did the flux_meter report in? One word only.')])]
qrun = astream(qmsgs, model='claude-sonnet-5', tools=[flux_meter])
qevs = [m async for m in qrun]
qrun.result['result']


'Gauss'

## Cleanup

Each run removes its own transcript. The scripted test leaves its temporary working directory and an empty session folder. The following cell removes those folders and the fake executable. It also deletes the session folder for the shared default cache project. Do not run that cleanup while another default run is active.

In [ ]:
shutil.rmtree(sess_dir(scratch), ignore_errors=True)
shutil.rmtree(sess_dir(work_dir()), ignore_errors=True)
shutil.rmtree(fake_cc.parent, ignore_errors=True)
shutil.rmtree(scratch, ignore_errors=True)

In [ ]:
#| hide
#| eval: false
import nbdev; nbdev.nbdev_export()